In [6]:
from pathlib import Path
import json
import pandas as pd

In [7]:
with open("cstc_book_titles.json", "r", encoding="utf-8") as f:
    included_book_titles = json.load(f)

included_titles_cleaned = [title.lower().strip('.') for title in included_book_titles]

with open("cstc_non_kids_books.json", "r", encoding="utf-8") as f:
    non_kids_books = json.load(f)

with open("cstc_sub_titles.json", "r", encoding="utf-8") as f:
    sub_titles = json.load(f)

In [9]:
stories = {}
counter = []

with open('cleaned_merged_fairy_tales_without_eos.txt', 'r') as f:
	for i, line in enumerate(f.readlines()):
		potential_title = line.strip('.;\n ').lower()
		if i == 0:
			current_story = ''
			current_title = potential_title

		if potential_title in included_titles_cleaned:
			stories[current_title] = current_story
			current_story = ''
			current_title = potential_title
		else:
			current_story += line
	
	stories[current_title] = current_story

kids_stories = {title: story for title, story in stories.items() if title not in non_kids_books}

In [10]:
counter = 0
filtered_kids_stories = {}
substory = False

for name, story in kids_stories.items():
	substory = False
	current_story = ''
	current_title = name
	
	# find substories
	for line in story.split('\n'):
		if line.startswith('BY') or line.startswith('ADAPTED'):
			continue

		if line in sub_titles:
			substory = True
			if current_story == '':
				current_title = line.strip('.;\n ').lower()
			else:
				filtered_kids_stories[current_title] = current_story
				current_title = line.strip('.;\n ').lower()
				current_story = ''
		else:
			current_story += line

	# clean-up
	if substory:
		filtered_kids_stories[current_title] = current_story
	else:
		filtered_kids_stories[name] = story

In [11]:
df_kids_stories = pd.DataFrame.from_dict(filtered_kids_stories, orient='index', columns=['value'])
df_kids_stories = df_kids_stories.reset_index().rename(columns={'index': 'title'})
df_kids_stories.rename(columns={'value':'story'}, inplace=True)
df_kids_stories

,title,story
0,the happy prince,"HIGH above the city, on a tall column, stood t..."
1,the emperor's new clothes,MANY years ago there was an emperor who was so...
2,the swineherd,THERE was once a poor prince who had a kingdom...
3,the real princess,THERE was once a prince who wanted to marry a ...
4,the shoes of fortune,I. A Beginning Every author has some peculiari...
...,...,...
228,the marvelous exploits of paul bunyan,Paul Bunyan \nScholars Say He is the Only Amer...
229,christmas every day and other stories,...
230,christmas every day,"The little girl came into her papa's study, as..."
231,the pumpkin-glory,The papa had told the story so often that the ...


In [19]:
df_kids_stories['story_length'] = df_kids_stories['story'].str.split().str.len()
df_kids_stories = df_kids_stories[df_kids_stories['story_length'] > 0]

df_short_kids = df_kids_stories[df_kids_stories['story_length'] < 10000].copy()
df_short_kids.drop(columns={'story_length'}, inplace=True)
df_short_kids.to_csv('cstc_preprocessed.csv')